In [1]:
import pandas as pd

financials = pd.read_csv("../data/processed/financials_long.csv")
review = pd.read_csv("../data/processed/universe_review.csv")

# Size gate thresholds from docs/investment_mandate.md ($000s)
REVENUE_MIN, REVENUE_MAX = 250_000, 1_000_000
EBITDA_MIN, EBITDA_MAX = 40_000, 100_000
MIN_FISCAL_YEAR = 2025   # require an annual report from the last ~12-18 months

# Long format -> wide format: one row per company and year, one column per variable
wide = financials.pivot_table(index=["cik", "ticker", "fiscal_year"], columns="variable",
                              values="value_000s", aggfunc="first").reset_index()

wide["ebitda"] = wide["operating_income"] + wide["d_and_a"]
print("Company-years:", len(wide))
wide.head()

Company-years: 1629


variable,cik,ticker,fiscal_year,accounts_receivable,acquisitions_paid,capex,cash,cfo,d_and_a,diluted_shares,finance_lease_current,finance_lease_noncurrent,goodwill,interest_expense,long_term_debt,operating_income,revenue,short_term_debt,stockholders_equity,ebitda
0,33185,EFX,2007,NaN,NaN,NaN,81600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,33185,EFX,2008,NaN,27400.0,NaN,58200.0,448100.0,155400.0,130400.0,NaN,NaN,NaN,71300.0,NaN,439000.0,NaN,NaN,NaN,594400.0
2,33185,EFX,2009,258700.0,196000.0,NaN,103100.0,418400.0,158800.0,127900.0,NaN,NaN,1943200.0,57000.0,990900.0,381800.0,NaN,183200.0,1601200.0,540600.0
3,33185,EFX,2010,262600.0,82600.0,NaN,119400.0,352600.0,167800.0,126500.0,NaN,NaN,1914700.0,56100.0,978900.0,430000.0,NaN,20700.0,1691400.0,597800.0
4,33185,EFX,2011,284400.0,127400.0,NaN,127700.0,408700.0,164900.0,123700.0,NaN,NaN,1961200.0,55100.0,966000.0,468600.0,NaN,47200.0,1702400.0,633500.0


In [2]:
# Each company's most recent year that has revenue
latest = (wide.dropna(subset=["revenue"])
              .sort_values("fiscal_year")
              .groupby("cik")
              .tail(1))


def size_gate(row):
    """Test one company's latest year against Ridgeline's size gate."""
    if row["fiscal_year"] < MIN_FISCAL_YEAR:
        return f"FAIL: no recent annual data (latest FY{int(row['fiscal_year'])})"
    if pd.isna(row["ebitda"]):
        return "FAIL: missing EBITDA inputs"
    if not (REVENUE_MIN <= row["revenue"] <= REVENUE_MAX):
        return "FAIL: revenue outside $250M-$1B"
    if not (EBITDA_MIN <= row["ebitda"] <= EBITDA_MAX):
        return "FAIL: EBITDA outside $40M-$100M"
    return "PASS"


latest["size_gate"] = latest.apply(size_gate, axis=1)

# Join back to every in-universe company, so none can silently disappear
in_universe = review[review["in_universe"]][["cik", "company_name", "sic_code"]]
gate = in_universe.merge(latest, on="cik", how="left")
gate["size_gate"] = gate["size_gate"].fillna("FAIL: no revenue data")

gate.to_csv("../data/processed/size_gate_results.csv", index=False)
print("Companies tested:", len(gate))
gate["size_gate"].value_counts()

Companies tested: 125


size_gate
FAIL: revenue outside $250M-$1B                87
FAIL: EBITDA outside $40M-$100M                17
FAIL: missing EBITDA inputs                     6
PASS                                            5
FAIL: no revenue data                           4
FAIL: no recent annual data (latest FY2024)     3
FAIL: no recent annual data (latest FY2017)     1
FAIL: no recent annual data (latest FY2023)     1
FAIL: no recent annual data (latest FY2019)     1
Name: count, dtype: int64

In [3]:
passed = gate[gate["size_gate"] == "PASS"].copy()
passed["revenue_$M"] = (passed["revenue"] / 1000).round(1)
passed["ebitda_$M"] = (passed["ebitda"] / 1000).round(1)
passed["ebitda_margin"] = (passed["ebitda"] / passed["revenue"]).round(3)

print("Companies passing the size gate:", len(passed))
passed[["ticker", "company_name", "sic_code", "fiscal_year",
        "revenue_$M", "ebitda_$M", "ebitda_margin"]].sort_values("revenue_$M", ascending=False)

Companies passing the size gate: 5


,ticker,company_name,sic_code,fiscal_year,revenue_$M,ebitda_$M,ebitda_margin
99,TISI,TEAM INC,7600.0,2025.0,896.5,48.2,0.054
77,ONT,"Onterris, Inc.",8742.0,2025.0,830.5,62.6,0.075
73,MG,"Mistras Group, Inc.",8711.0,2025.0,724.0,71.6,0.099
67,WLDN,"Willdan Group, Inc.",8711.0,2025.0,681.6,62.8,0.092
63,LQDT,LIQUIDITY SERVICES INC,7389.0,2025.0,476.7,45.5,0.095


In [4]:
# 1. Why are EBITDA inputs or revenue missing?
gaps = gate[gate["size_gate"].isin(["FAIL: missing EBITDA inputs", "FAIL: no revenue data"])].copy()
gaps["revenue_$M"] = (gaps["revenue"] / 1000).round(1)
gaps["has_operating_income"] = gaps["operating_income"].notna()
gaps["has_d_and_a"] = gaps["d_and_a"].notna()
print(gaps[["ticker", "company_name", "size_gate", "fiscal_year", "revenue_$M",
            "has_operating_income", "has_d_and_a"]].sort_values("revenue_$M", ascending=False).to_string())

# 2. Revenue failures: too small or too large?
rev_fail = gate[gate["size_gate"] == "FAIL: revenue outside $250M-$1B"]
print("\nRevenue too small (< $250M):", (rev_fail["revenue"] < REVENUE_MIN).sum())
print("Revenue too large (> $1B):  ", (rev_fail["revenue"] > REVENUE_MAX).sum())

# 3. EBITDA failures: too small or too large?
ebitda_fail = gate[gate["size_gate"] == "FAIL: EBITDA outside $40M-$100M"]
print("EBITDA too small (< $40M):  ", (ebitda_fail["ebitda"] < EBITDA_MIN).sum())
print("EBITDA too large (> $100M): ", (ebitda_fail["ebitda"] > EBITDA_MAX).sum())

    ticker                            company_name                    size_gate  fiscal_year  revenue_$M  has_operating_income  has_d_and_a
48    TNET                      TRINET GROUP, INC.  FAIL: missing EBITDA inputs       2025.0      5010.0                 False         True
92    CNDT                            CONDUENT Inc  FAIL: missing EBITDA inputs       2025.0      3042.0                 False         True
38    FTDR                         Frontdoor, Inc.  FAIL: missing EBITDA inputs       2025.0      2093.0                 False         True
111   BAER  Bridger Aerospace Group Holdings, Inc.  FAIL: missing EBITDA inputs       2025.0       122.8                 False         True
71    CASS            CASS INFORMATION SYSTEMS INC  FAIL: missing EBITDA inputs       2025.0       108.0                 False         True
119   ARAI                          Arrive AI Inc.  FAIL: missing EBITDA inputs       2025.0         0.1                 False         True
64     NaN          

In [5]:
# Companies failing on revenue, but whose EBITDA is within Ridgeline's affordable range
affordable = rev_fail[rev_fail["ebitda"].between(EBITDA_MIN, EBITDA_MAX)].copy()
affordable["revenue_$M"] = (affordable["revenue"] / 1000).round(1)
affordable["ebitda_$M"] = (affordable["ebitda"] / 1000).round(1)
affordable["ebitda_margin"] = (affordable["ebitda"] / affordable["revenue"]).round(3)

print("Fail revenue gate, but EBITDA is $40M-$100M:", len(affordable))
affordable[["ticker", "company_name", "sic_code", "revenue_$M", "ebitda_$M", "ebitda_margin"]].sort_values("revenue_$M")

Fail revenue gate, but EBITDA is $40M-$100M: 3


,ticker,company_name,sic_code,revenue_$M,ebitda_$M,ebitda_margin
84,RMR,RMR GROUP INC.,8742.0,182.7,53.3,0.292
68,QNST,"QUINSTREET, INC",7389.0,1293.7,60.6,0.047
79,ADV,Advantage Solutions Inc.,7389.0,3542.6,75.8,0.021


In [6]:
gaps[["ticker", "company_name", "size_gate", "fiscal_year", "revenue_$M",
      "has_operating_income", "has_d_and_a"]].sort_values("revenue_$M", ascending=False)

,ticker,company_name,size_gate,fiscal_year,revenue_$M,has_operating_income,has_d_and_a
48,TNET,"TRINET GROUP, INC.",FAIL: missing EBITDA inputs,2025.0,5010.0,False,True
92,CNDT,CONDUENT Inc,FAIL: missing EBITDA inputs,2025.0,3042.0,False,True
38,FTDR,"Frontdoor, Inc.",FAIL: missing EBITDA inputs,2025.0,2093.0,False,True
111,BAER,"Bridger Aerospace Group Holdings, Inc.",FAIL: missing EBITDA inputs,2025.0,122.8,False,True
71,CASS,CASS INFORMATION SYSTEMS INC,FAIL: missing EBITDA inputs,2025.0,108.0,False,True
119,ARAI,Arrive AI Inc.,FAIL: missing EBITDA inputs,2025.0,0.1,False,True
64,NaN,"Arlo Technologies, Inc.",FAIL: no revenue data,NaN,NaN,False,False
86,NaN,Iron Horse Acquisition II Corp.,FAIL: no revenue data,NaN,NaN,False,False
100,NaN,Melar Acquisition Corp. I/Cayman,FAIL: no revenue data,NaN,NaN,False,False
114,NaN,T3 Defense Inc.,FAIL: no revenue data,NaN,NaN,False,False


In [7]:
import json


def depreciation_tags(cik, fiscal_year):
    """List full-year 10-K tags mentioning depreciation or amortization for one fiscal year."""
    path = f"../data/raw/companyfacts/CIK{int(cik):010d}.json"
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    found = set()
    for tag, info in data["facts"].get("us-gaap", {}).items():
        if "Depreciation" in tag or "Amortization" in tag:
            for entries in info["units"].values():
                for e in entries:
                    if e.get("form") == "10-K" and "start" in e:
                        start, end = pd.Timestamp(e["start"]), pd.Timestamp(e["end"])
                        full_year = 350 <= (end - start).days <= 380
                        same_year = (end - pd.Timedelta(days=15)).year == fiscal_year
                        if full_year and same_year:
                            found.add((tag, e["val"] / 1000))
    return sorted(found)


for ticker in ["CTEV", "SCOR"]:
    row = gate[gate["ticker"] == ticker].iloc[0]
    print(f"\n{ticker} - {row['company_name']} (FY{int(row['fiscal_year'])})")
    for tag, value in depreciation_tags(row["cik"], int(row["fiscal_year"])):
        print(f"   {tag:75} {value:>12,.0f}")


CTEV - Claritev Corp (FY2025)
   AmortizationOfDebtDiscountPremium                                                -17,900
   AmortizationOfFinancingCostsAndDiscounts                                           5,478
   AmortizationOfIntangibleAssets                                                   343,757
   CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization               253,411
   Depreciation                                                                     101,669
   OperatingLeaseRightOfUseAssetAmortizationExpense                                   3,028

SCOR - COMSCORE, INC. (FY2025)
   AmortizationOfIntangibleAssets                                                     2,529
   CapitalizedComputerSoftwareAmortization1                                          22,300
   Depreciation                                                                      23,745
   FinanceLeaseRightOfUseAssetAmortization                                            3,713
   HostingArrange

In [8]:
check = financials[(financials["ticker"].isin(["CTEV", "SCOR"])) &
                   (financials["fiscal_year"] == 2025) &
                   (financials["variable"] == "d_and_a")]
check[["ticker", "value_000s", "tag"]]

,ticker,value_000s,tag
13055,CTEV,445426.0,Depreciation + AmortizationOfIntangibleAssets
16282,SCOR,26274.0,Depreciation + AmortizationOfIntangibleAssets
